# Phase 1: Data Preparation Pipeline

This notebook demonstrates the end-to-end data preparation pipeline for LISSclear.
It downloads Sentinel-2 L2A tiles (as proxies for LISS-IV if local data is unavailable), generates cloud masks, stacks temporal references, and extracts overlapping patches for diffusion training.

**Note for Colab:** Ensure you have mounted your Google Drive and set up the `.env` file with Sentinel Hub / Copernicus API credentials if downloading new data.

In [ ]:
!pip install -r ../requirements.txt -q
import sys; sys.path.append('..')
from pathlib import Path
from src.utils.config import default_config
from src.data.downloader import SentinelDownloader
from src.data.preprocessor import Preprocessor
from src.data.patch_extractor import PatchExtractor
import matplotlib.pyplot as plt

## 1. Configuration & Directories

In [ ]:
cfg = default_config
print(f"Raw Data Dir: {cfg.raw_data_dir}")
print(f"Processed Patches Dir: {cfg.processed_patches}")
cfg.raw_data_dir.mkdir(parents=True, exist_ok=True)
cfg.processed_patches.mkdir(parents=True, exist_ok=True)

## 2. Download Raw Data (Optional)
Skip this if you already have `.tif` tiles in `data/raw/`.

In [ ]:
# downloader = SentinelDownloader(cfg.raw_data_dir)
# tiles = downloader.download_time_series(
#     bbox=(72.0, 16.0, 73.0, 17.0), # Example: Western Ghats
#     start_date="2023-01-01",
#     end_date="2023-12-31",
#     max_cloud_cover=100.0 # We want both cloudy and clear tiles
# )

## 3. Preprocessing & Cloud Masking
Converts downloaded tiles to standardized 4-band (R,G,B,NIR) GeoTIFFs and generates binary cloud masks.

In [ ]:
preprocessor = Preprocessor(
    input_dir=cfg.raw_data_dir,
    output_dir=cfg.raw_data_dir / "processed_tiles",
)
processed_files = preprocessor.run_pipeline()

## 4. Extract Patches & Build Temporal Stacks
Finds cloudy tiles, pairs them with cloud-free references from the same location, and extracts 256x256 training patches.

In [ ]:
extractor = PatchExtractor(
    patch_size=cfg.patch_size,
    stride=cfg.patch_stride,
    min_cloud_pct=0.1,  # Require at least 10% cloud cover to be a target
    max_cloud_pct=0.9,  # Ignore fully cloudy patches (no context)
)

# This will output pairs to data/processed/patches/
# n_patches = extractor.extract_dataset(
#     input_dir=cfg.raw_data_dir / "processed_tiles",
#     output_dir=cfg.processed_patches
# )
# print(f"Extracted {n_patches} training patches.")